# 🏦 Stage 1 — Non-Instruction Fine-Tuning
## Finance FAQ Assistant · Qwen2.5-0.5B · Google Colab T4

---

### 🗺️ What is this notebook about?

Before we teach an AI model to *answer* finance questions, we need it to *speak the language of finance*.

> **Analogy:** Think of a brilliant new hire who has never worked in a bank.
> Before they answer customer queries, you make them read finance textbooks cover-to-cover.
> That's exactly what this notebook does — but for a language model.

We feed the model **57 raw finance paragraphs** — no questions, no answers, just plain domain text —
and train it to predict the next word. This forces it to absorb:
- Finance vocabulary: EMI, SIP, NAV, TDS, QLoRA, moratorium, collateral …
- How concepts relate: credit utilization → credit score → loan approval
- The style and tone used in professional finance writing

---

### 🧱 The Full 3-Stage Pipeline (Big Picture)

```
Qwen2.5-0.5B (pre-trained on general internet text)
        │
        ▼ ← YOU ARE HERE
┌─────────────────────────────────────────────────────┐
│  STAGE 1 — Non-Instruction Fine-Tuning              │
│  Data   : 57 raw finance paragraphs                 │
│  Method : next-token prediction (no Q&A format)     │
│  Goal   : domain vocabulary & writing style         │
│  Output : finance_stage1_adapter/                   │
└─────────────────────────────────────────────────────┘
        │
        ▼  (loaded by notebook 2)
┌─────────────────────────────────────────────────────┐
│  STAGE 2 — Instruction Fine-Tuning (SFT)            │
│  Data   : 104 question-answer pairs                 │
│  Method : SFTTrainer + Qwen2.5 chat template        │
│  Goal   : learn to answer finance questions          │
│  Output : finance_sft_adapter/                      │
└─────────────────────────────────────────────────────┘
        │
        ▼  (loaded by notebook 3)
┌─────────────────────────────────────────────────────┐
│  STAGE 3 — DPO Preference Alignment                 │
│  Data   : 51 chosen / rejected pairs                │
│  Method : DPOTrainer · beta=0.1 · lr=5e-6           │
│  Goal   : prefer complete, safe, professional answers│
│  Output : finance_dpo_adapter/                      │
└─────────────────────────────────────────────────────┘
```

---

### ⚙️ Technology Choices at a Glance

| What | Choice | Why |
|---|---|---|
| Base model | `Qwen2.5-0.5B` | 500M params — fits free Colab T4 (≈16 GB VRAM) |
| Loading precision | 4-bit (QLoRA) | Cuts base model VRAM from ~1 GB → ~0.3 GB |
| Fine-tuning method | LoRA adapters | Only 1.75% of weights trained; saves ~98% memory |
| Training speed | Unsloth | ~2× faster than vanilla HuggingFace on the same GPU |
| Trainer | SFTTrainer (TRL) | Ready-made causal-LM training loop |

**⏱️ Expected runtime on free Colab T4:** ≈ 2 minutes  
**💾 Output size:** `finance_stage1_adapter/` ≈ 33 MB (adapters only, not the full base model)


## 📦 Step 1 — Install Dependencies

We install four groups of packages. Run this cell first every time you open a new Colab session.

| Package | Role |
|---|---|
| `unsloth` | Speed-optimised fine-tuning wrapper — 2× faster than plain HuggingFace on T4 |
| `xformers` | Memory-efficient attention kernels that reduce VRAM usage during training |
| `trl` | Provides `SFTTrainer` (Stages 1 & 2) and `DPOTrainer` (Stage 3) |
| `peft` | LoRA/QLoRA adapter framework — attaches trainable matrices to frozen layers |
| `accelerate` + `bitsandbytes` | 4-bit quantization and distributed-training utilities |
| `datasets` | HuggingFace dataset loading and `.map()` preprocessing |
| `sentencepiece` | Required tokenizer support for Qwen2.5 |

> **`%%capture`** hides the long pip output to keep the notebook clean.
> The `torch.cuda.get_device_capability()` call is used by Unsloth internally
> to select the right CUDA kernels for the specific GPU detected.

> ⚠️ **If you restart the Colab runtime, re-run this cell before anything else.**


In [1]:
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()

!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" trl peft accelerate bitsandbytes
!pip install datasets sentencepiece

## 📂 Step 2 — Load Raw Domain Text

We read `non_instruction_data.txt` — a file with **57 hand-written finance paragraphs**.

Topics covered:
> savings & current accounts · fixed deposits · recurring deposits · KYC ·
> credit scores · credit cards · personal/home/auto loans · EMI · prepayment ·
> APR · mutual funds · SIPs · NAV · demat accounts · term & health insurance ·
> TDS · income tax · capital gains · emergency funds · fraud protection

**Why plain text and not Q&A pairs?**
At this stage we do **not** want the model to learn to answer questions.
We want it to absorb domain vocabulary the same way the original pre-training
absorbed general English — by reading and predicting text, one word at a time.

**How to get the file into Colab:**
```python
# Option A — upload directly
from google.colab import files
files.upload()          # select non_instruction_data.txt

# Option B — clone the repo
!git clone https://github.com/mayankchugh-learning/domain-ai-assistant-finetuning
```
The code below assumes the file is at `/content/data/non_instruction_data.txt`
after cloning or after uploading to that path.

> 📌 **Expected output:** `Loaded 57 paragraphs`


In [2]:
# Upload data/non_instruction_data.txt to Colab, or mount Drive / clone the repo
# from google.colab import files
# uploaded = files.upload()  # select non_instruction_data.txt

DATA_PATH = "/content/data/non_instruction_data.txt"  # adjust path if using Drive/Git clone

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]
print(f"Loaded {len(paragraphs)} paragraphs")
print(paragraphs[0][:300])

Loaded 57 paragraphs
A savings account is a deposit account held at a bank that earns interest on the balance. It is meant for individuals who want to set aside money safely while earning a modest return. Most savings accounts allow limited withdrawals per month and may charge a fee if the minimum balance is not maintai


## 🧹 Step 3 — Clean and Chunk the Text

Two steps happen here before the text goes into training.

### Step A — Cleaning
```python
re.sub(r"\s+", " ", text)        # collapse multiple spaces/tabs/newlines → single space
re.sub(r"[^\x00-\x7F]+", "", text)  # remove non-ASCII characters (stray symbols, smart quotes)
```
Paragraphs with fewer than **15 words** are dropped — too short to carry useful signal.

### Step B — Chunking
`max_seq_length = 512` tokens is the model's context window.
Each of our paragraphs is already ~80–120 words, well inside that limit.
`MAX_CHUNK_WORDS = 150` is a safety cap for any unusually long paragraphs —
those get split into two smaller chunks automatically.

**Why chunk at the paragraph level rather than sentences?**
A full paragraph preserves the coherent reasoning chain within a finance concept
(e.g. "what is X → why it matters → how to use it").
Shorter sentence fragments would lose that context, giving the model weaker signal.

> 📌 **Expected output:** `57 paragraphs after cleaning` → `Final chunk count: 57`
> (All paragraphs pass the 15-word filter, so count stays at 57.)


In [3]:
import re

def clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"[^\x00-\x7F]+", "", text)  # strip non-ASCII noise
    return text

cleaned_paragraphs = [clean_text(p) for p in paragraphs if len(p.split()) > 15]
print(f"{len(cleaned_paragraphs)} paragraphs after cleaning/length filter")

# Chunking strategy: each paragraph is already a coherent ~80-120 word chunk,
# which is a good size for causal-LM continuation training on a 0.5B model.
MAX_CHUNK_WORDS = 150

def chunk_paragraph(p, max_words=MAX_CHUNK_WORDS):
    words = p.split()
    if len(words) <= max_words:
        return [p]
    return [" ".join(words[i:i+max_words]) for i in range(0, len(words), max_words)]

chunks = []
for p in cleaned_paragraphs:
    chunks.extend(chunk_paragraph(p))

print(f"Final chunk count: {len(chunks)}")

57 paragraphs after cleaning/length filter
Final chunk count: 57


## 🤖 Step 4 — Load the Base Model with Unsloth

### What is Qwen2.5-0.5B?
Qwen2.5-0.5B is a **500-million-parameter** large language model from Alibaba's Qwen team.
Pre-trained on a massive general corpus — news, books, code, web pages.
It speaks general English well, but has **no particular depth in finance**.
That gap is exactly what our fine-tuning closes.

### The three key loading parameters

| Parameter | Value | What it controls |
|---|---|---|
| `max_seq_length` | `512` | Maximum tokens the model sees at once during training |
| `dtype` | `None` (auto) | T4 supports float16; Unsloth auto-selects the right precision |
| `load_in_4bit` | `True` | **QLoRA**: compress base weights from 16-bit → 4-bit, saving ~75% VRAM |

### What is QLoRA in plain English?
Normally, loading a 0.5B model in 16-bit float takes ≈1 GB of GPU memory.
4-bit quantization stores each weight in 4 bits instead of 16, cutting that to ≈0.3 GB.
The quality loss is negligible for the frozen forward pass — and since we only update
tiny LoRA adapters (not the base weights), the training quality is preserved.

> 📌 **Expected output:** Unsloth prints the GPU name (Tesla T4) and available VRAM (~14.56 GB).
> Loading completes in about 30 seconds.


In [4]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 512
dtype = None  # auto-detect (bfloat16 on T4-supported hardware, float16 otherwise)
load_in_4bit = True  # QLoRA-style 4-bit loading to fit comfortably on T4

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

## 🔧 Step 5 — Apply LoRA Adapters

### Why not update all 500M weights?
Full fine-tuning needs gradients and Adam optimizer states for every parameter —
that's 3–4× the model size in extra VRAM, far beyond what a T4 can hold.

### What LoRA does instead

LoRA (Low-Rank Adaptation) **freezes all original weights** and injects tiny
trainable matrices alongside specific layers:

```
Original weight W  →  FROZEN (not updated, no gradient stored)
      +
LoRA adapter: W_update ≈ B × A
  • A has shape [r, d_in]   — r rows, d_in columns
  • B has shape [d_out, r]  — d_out rows, r columns
  • r = 16 (the "rank")     — a tiny inner dimension

Effective weight = W + B×A  (same math, 98.25% less memory)
```

### Which layers get adapters?
```python
target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",   # Attention — how the model focuses on text
    "gate_proj", "up_proj", "down_proj"        # MLP — how the model transforms information
]
```
These 7 projection layers are the weight matrices most responsible for
domain-specific knowledge and writing style — exactly what we want to update.

### Hyperparameter rationale

| Parameter | Value | Reasoning |
|---|---|---|
| `r` (rank) | `16` | Capacity of the adapter. Low rank = few params. 16 is a proven sweet spot for small domain datasets. |
| `lora_alpha` | `16` | Scaling factor. `alpha/r = 1.0` means the adapter updates at full scale without extra amplification. |
| `lora_dropout` | `0.05` | 5% dropout on adapter weights. Light regularisation to prevent overfitting on only 57 paragraphs. |
| `bias` | `"none"` | Don't add trainable bias terms — negligible quality effect, minor memory saving. |
| `use_gradient_checkpointing` | `"unsloth"` | Recomputes activations during backward pass instead of storing them. Saves ~40% VRAM at the cost of ~15% compute. |
| `random_state` | `42` | Fixes the random seed for reproducible adapter initialisation. |

> 📌 **Expected output:**
> `trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7502`
> We train just **1.75%** of the total weights — yet this is enough to meaningfully
> shift the model's domain behaviour.


In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                 # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.7.2 patched 24 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


## 📊 Step 6 — Build the Training Dataset

We convert our 57 clean text chunks into a HuggingFace `Dataset` object that
`SFTTrainer` can consume directly.

### Why append the EOS token?
```python
texts = [chunk + tokenizer.eos_token for chunk in chunks]
```
The **EOS (End-Of-Sequence) token** marks where each paragraph ends.
Without it, the model would not learn clean paragraph boundaries and might
blend concepts from different paragraphs during generation.

### What does `Dataset.from_dict` do?
It wraps the list of strings in a simple `{"text": [...]}` dictionary and
creates a HuggingFace Dataset object. This gives us access to `.map()`,
batching, and the `dataset_text_field="text"` argument in `SFTTrainer`.

> 📌 **Expected output:** `Dataset({features: ['text'], num_rows: 57})`


In [6]:
from datasets import Dataset

# Add EOS so the model learns natural paragraph boundaries
texts = [c + tokenizer.eos_token for c in chunks]
raw_dataset = Dataset.from_dict({"text": texts})
print(raw_dataset)

Dataset({
    features: ['text'],
    num_rows: 57
})


## 🏋️ Step 7 — Configure the SFT Trainer

`SFTTrainer` from the TRL library handles causal language modelling training.
Even though this is *non-instruction* training (plain text, no Q&A format),
`SFTTrainer` is perfect here — we just point it at the `"text"` column and
it runs standard next-token-prediction loss.

### Training hyperparameters — every argument explained

| Argument | Value | Why |
|---|---|---|
| `per_device_train_batch_size` | `4` | Process 4 text chunks per GPU step |
| `gradient_accumulation_steps` | `4` | Accumulate gradients over 4 steps → effective batch = **16** |
| `warmup_steps` | `10` | Ramp the learning rate up slowly for the first 10 steps. Prevents unstable spikes early in training. |
| `num_train_epochs` | `3` | Read through all 57 paragraphs 3 times. Enough to absorb vocabulary without memorising verbatim. |
| `learning_rate` | `2e-4` | Standard LoRA LR. Higher than full FT because only 1.75% of weights are updated. |
| `fp16` / `bf16` | auto | Half-precision arithmetic halves VRAM for activations. `is_bfloat16_supported()` picks the right one for the GPU. |
| `logging_steps` | `5` | Print training loss every 5 steps to track progress. |
| `optim` | `"adamw_8bit"` | 8-bit Adam — same quality as standard Adam but 4× less VRAM for optimizer states. |
| `weight_decay` | `0.01` | L2 regularisation on adapter weights. Prevents the small adapter from memorising the 57 paragraphs. |
| `lr_scheduler_type` | `"linear"` | Learning rate decays linearly from `2e-4` to `0` over all training steps. |
| `packing` | `False` | Don't pack multiple short texts into one sequence. Keeps paragraph boundaries clean. |

### Why gradient accumulation?
The T4 can't fit 16 samples in VRAM at once.
Instead: process 4 → accumulate gradient → process 4 more → accumulate → × 4 → update weights.
**The math is identical to a batch of 16**, but memory usage is that of a batch of 4.


In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=raw_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs_stage1",
        save_strategy="no",
        report_to="none",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False}
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/57 [00:00<?, ? examples/s]

## 🚀 Step 8 — Run Training

This single call launches the full training loop — loading data, forward pass,
loss computation, backward pass, weight update — for all 3 epochs.

### What to watch in the output
The trainer prints a loss value every 5 steps (controlled by `logging_steps`).

**Healthy Stage 1 training looks like:**

| Step | Loss | Interpretation |
|---|---|---|
| 1–5 | ~2.26 | Starting loss — LoRA adapters are randomly initialised |
| 10 | ~1.99 | Model is learning to predict finance text |
| Final | ~2.07 | Stable convergence — some fluctuation is normal with only 57 samples |

**Is a final loss of ~2.0 good?**
Yes, for this task. We are not trying to get loss to 0 (that would mean memorising the
57 paragraphs verbatim). We just want the model to shift its probability distribution
toward finance language. A loss around 2.0 means the perplexity is about e² ≈ 7.4 —
the model assigns meaningful probability to the correct next finance token.

> 📌 **Expected training time:** ~65 seconds on Colab T4
> **Samples/second:** ~2.6 | **Total steps:** ~12 (57 samples ÷ batch 16 × 3 epochs)


In [8]:
trainer_stats = trainer.train()
print(trainer_stats)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 57 | Num Epochs = 3 | Total steps = 12
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
5,2.261713
10,1.997996


TrainOutput(global_step=12, training_loss=2.0658821066220603, metrics={'train_runtime': 56.4081, 'train_samples_per_second': 3.031, 'train_steps_per_second': 0.213, 'total_flos': 22230597703680.0, 'train_loss': 2.0658821066220603, 'epoch': 3.0})


## 💾 Step 9 — Save the Stage 1 Adapter

```python
model.save_pretrained("finance_stage1_adapter")    # saves adapter weights
tokenizer.save_pretrained("finance_stage1_adapter") # saves tokenizer files
```

### What gets saved — and what doesn't
`save_pretrained` saves **only the LoRA adapter** (~33 MB), not the full base model (~1 GB).

The `finance_stage1_adapter/` folder contains:
```
adapter_config.json          ← records rank=16, alpha=16, target_modules
adapter_model.safetensors    ← the actual trained A and B weight matrices
tokenizer.json               ← vocabulary
tokenizer_config.json        ← chat template and special token config
special_tokens_map.json      ← EOS, PAD, BOS token IDs
```

**Key insight:** Anyone with `Qwen2.5-0.5B` + this 33 MB folder can fully
reconstruct Stage 1's model. You never need to redistribute the 1 GB base model.


In [9]:
model.save_pretrained("finance_stage1_adapter")
tokenizer.save_pretrained("finance_stage1_adapter")
print("Stage 1 adapter saved to finance_stage1_adapter/")

# Optional: zip and download, or push to Hugging Face Hub for use in Stage 2
# from huggingface_hub import HfApi
# model.push_to_hub("your-username/finance-stage1-adapter", token="hf_...")

Unsloth: Restored added_tokens_decoder metadata in finance_stage1_adapter/tokenizer_config.json.


Stage 1 adapter saved to finance_stage1_adapter/


## ☁️ Step 10 — Upload Adapter (Choose: Download or Hugging Face Hub)

### Option A — Download to your computer
Run the cell below to zip and download `finance_stage1_adapter/` directly.
You'll re-upload it at the start of notebook 2.

> ⚠️ **Important for notebook chaining:**
> Stage 2 (`instruction_finetuning.ipynb`) loads this adapter as its starting point.
> Save it before closing this Colab session — Colab VMs are ephemeral and
> files disappear when the session ends.


In [10]:
from google.colab import files
import shutil

shutil.make_archive("finance_stage1_adapter", "zip", "finance_stage1_adapter/")
files.download("finance_stage1_adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Option B — Push directly to Hugging Face Hub *(recommended for seamless chaining)*

Pushing to Hub means notebook 2 can download the adapter automatically with one line,
without you manually uploading a zip file. Run cells B1 → B4 in order.

**B1 — Install / update the Hub CLI**


In [12]:
!pip install -q -U huggingface_hub

**B2 — Authenticate with your Hugging Face WRITE token**

Get your token at → https://huggingface.co/settings/tokens
Select **"Write"** access. The token is stored in an environment variable for this session only.


In [14]:
import getpass, os
os.environ["HF_TOKEN"] = getpass.getpass("Paste your Hugging Face WRITE access token: ")

Paste your Hugging Face WRITE access token: ··········


**B3 — Verify the token is set**


In [15]:
print("Token set:", bool(os.environ.get("HF_TOKEN")))

Token set: True


**B4 — Create the repo and upload the adapter folder**

`exist_ok=True` means this is safe to re-run — it won't fail if the repo already exists.
`delete_patterns=["*"]` clears the repo before uploading, so you always get a clean state.


In [16]:
from huggingface_hub import HfApi
repo_id = "mayankchugh-learning/finance_stage1_adapter"
api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True, private=False)
api.upload_folder(repo_id=repo_id, repo_type="model", folder_path="finance_stage1_adapter", delete_patterns=["*"], commit_message="Update Stage 1 adapter")
print("Uploaded to https://huggingface.co/" + repo_id)

Uploaded to https://huggingface.co/mayankchugh-learning/finance_stage1_adapter


## 🧪 Step 11 — Test the Stage 1 Model (Free-Form Continuation)

### ⚠️ Important: what we are NOT testing
We are **not** asking the model questions and expecting direct answers.
Stage 1 only teaches domain *fluency* — not Q&A *behaviour*.
That comes in Stage 2.

### What we ARE testing
We give the model the start of a finance sentence and let it complete it freely.
This verifies whether it has absorbed finance vocabulary and reasoning patterns.

### Generation settings used here
```python
do_sample=True,      # enable random sampling (for more varied completions)
temperature=0.7,     # moderate creativity — not too random, not too repetitive
top_p=0.9            # nucleus sampling: only consider tokens in top 90% probability mass
```
This is different from evaluation in Stage 2/3, where we use `do_sample=False`
for deterministic, reproducible answers.

### What healthy Stage 1 output looks like
| Prompt | What to look for |
|---|---|
| `"A fixed deposit is"` | Mentions interest rate, fixed tenure, bank |
| `"The interest-free period on a credit card"` | Mentions billing cycle, due date, full payment |
| `"To improve your credit score, you should"` | Mentions timely payments, credit utilisation |
| `"A SIP allows an investor to"` | Mentions regular intervals, mutual fund, averaging |

✅ Finance terminology used correctly → Stage 1 succeeded  
⚠️ Occasional factual imprecision is fine — the model only saw 57 paragraphs.  
Stage 2 corrects this with 104 verified Q&A pairs.

---

### 🔗 What comes next?
`finance_stage1_adapter/` is the input to **notebook 2 — instruction_finetuning.ipynb**.
Stage 2 continues training from this domain-adapted checkpoint, giving it a head-start
over beginning from the raw base model.


In [11]:
FastLanguageModel.for_inference(model)

test_prefixes = [
    "A fixed deposit is",
    "The interest-free period on a credit card",
    "To improve your credit score, you should",
    "A SIP allows an investor to",
]

for prefix in test_prefixes:
    inputs = tokenizer(prefix, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=60, do_sample=True, temperature=0.7, top_p=0.9)
    print("PROMPT:", prefix)
    print("CONTINUATION:", tokenizer.decode(out[0], skip_special_tokens=True))
    print("-" * 80)

Both `max_new_tokens` (=60) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

PROMPT: A fixed deposit is
CONTINUATION: A fixed deposit is a type of savings account that earns interest on a fixed percentage rate, typically stated as a percentage of the initial deposit amount, compounded monthly. Unlike other savings accounts, fixed deposits do not have a maturity date, meaning they are withdrawn early without penalty.
--------------------------------------------------------------------------------


Both `max_new_tokens` (=60) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: The interest-free period on a credit card
CONTINUATION: The interest-free period on a credit card typically expires after a certain number of days, after which the full balance is typically due.
--------------------------------------------------------------------------------


Both `max_new_tokens` (=60) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT: To improve your credit score, you should
CONTINUATION: To improve your credit score, you should regularly review your credit report and make payments on time.
--------------------------------------------------------------------------------
PROMPT: A SIP allows an investor to
CONTINUATION: A SIP allows an investor to purchase shares of a company at a predetermined price, and receive dividends on those shares when the company earns profit.
--------------------------------------------------------------------------------
